In [1]:
import torch
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from pathlib import Path

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

PyTorch version: 2.14.1+cpu
Torchvision version: 0.29.1+cpu


In [2]:
DATASET_DIR = Path("../dataset")

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

print("Train:", TRAIN_DIR.resolve())
print("Validation:", VAL_DIR.resolve())
print("Test:", TEST_DIR.resolve())

Train: D:\deepfake\dataset\train
Validation: D:\deepfake\dataset\validation
Test: D:\deepfake\dataset\test


In [3]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
# Load the datasets

train_dataset = ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform
)

val_dataset = ImageFolder(
    root=VAL_DIR,
    transform=val_test_transform
)

test_dataset = ImageFolder(
    root=TEST_DIR,
    transform=val_test_transform
)

print("Classes:", train_dataset.classes)
print("Class to index:", train_dataset.class_to_idx)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Classes: ['fake', 'real']
Class to index: {'fake': 0, 'real': 1}

Dataset sizes:
Train: 1428
Validation: 306
Test: 307


In [5]:
# Create DataLoaders

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 90
Validation batches: 20
Test batches: 20


In [6]:
# Check one batch

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

Image batch shape: torch.Size([16, 3, 224, 224])
Label batch shape: torch.Size([16])
Labels: tensor([0, 0, 0, 1, 0, 1, 1, 1, 0, 1])


In [7]:
# Create EfficientNet-B0 model

import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Load pretrained EfficientNet-B0
weights = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(weights=weights)

# Check the original classifier
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [8]:
# Get the number of input features
in_features = model.classifier[1].in_features

# Replace the original 1000-class classifier
model.classifier[1] = nn.Linear(in_features, 2)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)


In [9]:
print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

In [10]:
model = efficientnet_b0(weights=weights)

In [11]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [12]:
device = torch.device("cpu")

model = model.to(device)

print("Using device:", device)

Using device: cpu


In [13]:
# Define loss function and optimizer

import torch.optim as optim

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [14]:
lr=0.0001

In [15]:
# Freeze the EfficientNet backbone

# Freeze EfficientNet feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# Keep classifier trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# Count trainable parameters
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model.parameters()
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 5288548
Trainable parameters: 1281000


In [16]:
# Re-create optimizer

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [17]:
# Test one batch before training

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

# Forward pass
outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Labels shape:", labels.shape)
print("Outputs:")
print(outputs[:5])

Input shape: torch.Size([16, 3, 224, 224])
Output shape: torch.Size([16, 1000])
Labels shape: torch.Size([16])
Outputs:
tensor([[-0.8290, -0.7588, -0.4281,  ..., -1.0026, -1.0714,  0.4479],
        [-0.1647, -2.4761,  0.3113,  ..., -1.8172, -1.7037, -0.7020],
        [ 0.5250, -1.6998, -0.3220,  ..., -0.6555,  0.5970, -1.7618],
        [-1.9272, -0.8894, -0.6904,  ..., -0.4608, -0.3840, -0.2097],
        [-0.7832,  0.4038, -1.8444,  ..., -0.4701, -0.3073,  1.3491]],
       grad_fn=<SliceBackward0>)
